# ThinCurr Python Example: DIII-D VDE Reconstruction using JAMfit 

In this example we demonstrate how to use JAMfit.py for performing a filament reconstruction for a DIII-D vde example. This notebook is a continuation of the vde_diiid_create.ipynb example notebook and will be using the data files generated from that notebook. 

In this notebook we will be using and comparing the different types of numerical solvers present in JAMfit. 

## Loading Relevant Libraries 

To load the JAMfit ThinCurr module, you will need to define where ThinCurr is located. This can be done through the PYTHONPATH enviornment variable or within the script using sys.path.append() as shown below, where we look for the environment variable OFT_ROOTPATH to point the path to where the OpenFUSIONToolkit is installed. If this enviornment variable is not defined, you can do so within the script as well by setting:
 
 
 ```os.environ["OFT_ROOTPATH"] = \path\to\OpenFUSIONToolkit\install_release```

In [1]:
import os 
import sys
import numpy as np
import re
import matplotlib.pyplot as plt
from pathlib import Path as pathlib_create 
import xml.etree.ElementTree as ET
from collections import OrderedDict
from matplotlib.path import Path
import matplotlib.tri as tri 
from IPython.display import clear_output
from matplotlib.colors import SymLogNorm
from scipy.interpolate import griddata





## DELETE THIS BEFORE LAUNCHING ## 
home_dir = os.path.expanduser("~")
oft_root_path = os.path.join(home_dir, "OpenFUSIONToolkit/install_release")
os.environ["OFT_ROOTPATH"] = oft_root_path
## END DELETE BLOCK ## 

thincurr_python_path = os.getenv('OFT_ROOTPATH')
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit._core import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.JAMfit import JAMfit, get_inside_limiter_pts

from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh

myOFT = OFT_env(nthreads=4)


#----------------------------------------------
             ____  ____________
            / __ \/ ____/_  __/
           / / / / /_    / /
          / /_/ / __/   / /
          \____/_/     /_/

Base release:        v1.0.0-beta7
Development branch:  clean-jamfit-pull-request
Revision id:         b4d76c7

Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    4

Linear Algebra backend: native

#----------------------------------------------



## Intializing JAMfit Object and Loading in Input Data

Here we define the relevant input files necessary for JAMfit reconstructions, namely the thincurr mesfile (`thincurr_meshfile`), the xml file with the shaping and solenoid coil as well as the plasma filament definitions (`xml_name`), and the path of the reduced ThinCurr object that was created in the previous notebook (`reduced_filename`). 

JAMfit reconstructions also require the following data: 

1. Magnetic sensor data corresponding to the same magnetic sensors that were intialized when creating the intial JAMfit object. Note that they also must be in the same order as when they were initalized/created.
2. Time array, for reconstructions that are over multiple time slices. 
3. Total plasma current 
4. Coil currents - for currents flowing in the coils.

In [2]:
meshfile_thincurr = "DIIID_meshfile_thincurr.h5" 
meshfile_tokamaker = "DIIID_tokamaker_JAMfit_model.h5"
xml_name = "DIIID_JAMfit_wfil_ex.xml" 
reduced_filename = "DIIID_vde_reduced_model_ex.h5" 
jam_obj = JAMfit(xml_name, meshfile_thincurr, meshfile_tokamaker, B0=1.91503608, R0=1.69550002, oft_env=myOFT)
jam_obj.initialize_reduced_model(reduced_filename)

magnetic_data = np.load('sensor_measurements.npy')
time_array_sensors = np.load('time_array_sensors.npy')
time_array_coils = np.load('time_array_coilcurr.npy')
totalip = np.load('totalip.npy')
coilcurrs = np.load('coil_currs.npy')

NameError: name 'meshfile_tokamaker' is not defined

### Helper Functions

This helper function serves to grab the r and z points of the plasma filaments. 

In [ ]:
def get_grid_from_xml(xml_path): ## for this to work the filaments cannot be named! 
    tree = ET.parse(xml_path)
    root = tree.getroot()

    r_grid = []
    z_grid = [] 
    for coil_set in root.iter('coil_set'):
        if coil_set.get('name') is None:  # unnamed only
            for coil in coil_set.findall('coil'):
                r, z = [float(v.strip()) for v in coil.text.split(',')]
                r_grid.append(r)
                z_grid.append(z)

    return r_grid, z_grid #

## General Variables Used for All Solvers

These input variables tend to be used by all solvers, where ```rgrid``` and ```zgrid``` are the R, Z locations of the filament points. ```num_non_fil_coils``` are the number of shaping coils present and whose flux contributions we need to subtract from the measured magnetics sensor signals within our algorithms. ```rms_global``` and ```sigma``` relate to the average standard deviation of each magnetic sensor signal which is used to weight each sensor equally. This is such that one sensor signal does not dominate over others within our solvers. ```num_sensors``` is the number of physical sensors that measure magnetic sensor signals (Mirnovs, flux loops etc.)

In [ ]:
rgrid, zgrid = get_grid_from_xml(xml_name)
num_non_fil_coils = coilcurrs.shape[1]
rms_global = np.sqrt(np.mean(magnetic_data**2))
sigma = np.maximum(0.05 * np.abs(magnetic_data).mean(axis=0), 0.02 * rms_global)
num_sensors = magnetic_data.shape[1]
points = np.column_stack((rgrid, zgrid))
num_filaments = len(points)


## Running JAMfit's LSTSQ Reconstruction Algorithm 

The simplest solver bundled in JAMfit is the direct least squares solver that has tikonov regularization. 

In the following cell, we show how to use the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.run_reconstruction_lstsq "run_reconstruction_lstsq()" in a loop over timesteps. 

>**Note:** that in this example, the ideal ip weighting value ```lam``` as well as the regularization factors ```reg_factor_fil``` and ```reg_factor_wall``` have already been found manually. In practical applications, sweeping through reg factors and lam weighting factors and comparing to the residual is one way for determining the ideal regularization factor. 

In [ ]:
solution_lstsq_fil_tidx = [] 
solution_lstsq_wall_tidx = [] 
residual_lstsq_tidx = [] 

lam = 4E2 #this is our ip weighting 

reg_factor_fil = 4E-4 # this is an L2 tikanov weighting for the plasma filaments 
reg_factor_wall = 1E-6 # this is an L2 tikanov weighting for the wall 

for t_idx in range(len(time_array_sensors)):
    psi_at_time = magnetic_data[t_idx, :]
    ip_at_time = totalip[t_idx]
    coil_curr_at_time = coilcurrs[t_idx, :]
    solution_fil, solution_wall, residual, Ax, B = jam_obj.run_reconstruction_lstsq(psi_at_time, ip_at_time, num_non_fil_coils, coil_curr_at_time, lam, sigma, reg_factor_fil ,reg_factor_wall, num_sensors)
    solution_lstsq_fil_tidx.append(solution_fil) 
    solution_lstsq_wall_tidx.append(solution_wall) 
    residual_lstsq_tidx.append(residual)


solution_lstsq_fil_tidx = np.array(solution_lstsq_fil_tidx)
solution_lstsq_wall_tidx = np.array(solution_lstsq_wall_tidx)
residual_lstsq_tidx = np.array(residual_lstsq_tidx)


### Plotting LSTSQ Results
The cell below is for grabbing the limiter for plotting convience, as well as grabbing ```coil_dict``` which we will be using in the post-processing routines. 

In [ ]:
mygs = TokaMaker(myOFT)
mesh_pts,mesh_lc,mesh_reg,coil_dict,cond_dict = load_gs_mesh(meshfile_tokamaker)
mygs.setup_mesh(mesh_pts, mesh_lc, mesh_reg)
mygs.setup_regions(cond_dict=cond_dict,coil_dict=coil_dict)
mygs.setup(order = 2, F0=1*1) # here we put an arbitrary multiplication for F0 as we only need the limiter 
limiter = mygs.lim_contour
clear_output() 

Here we plot the relevant filament jtor solutions. We can see that the lstsq tikanov solver does struggle. This is especially true when there is a large number of filaments. The lstsq tikanov solver also cannot solve if the number of filaments is greater than the number of sensors as the problem would become underdetermined. In our case we have 101 filaments and 117 magnetic sensors, which makes the lstsq tikanov solver not as robust. 

In [ ]:
# =================================
# Plotting/Processing results 
# =================================

plt.figure()
plt.plot(time_array_sensors, totalip, label='Total Ip')
plt.plot(time_array_sensors, np.sum(solution_lstsq_fil_tidx, axis=1), 'k--', label='Reconstructed Ip')
plt.xlabel('Time')
plt.ylabel('Ip [A]')
plt.ylim(0, max(np.max(totalip), np.max(np.sum(solution_lstsq_fil_tidx, axis=1))) * 1.1)
plt.legend()
plt.show()

print("Ip error percentage: ", np.mean(np.abs(totalip - np.sum(solution_lstsq_fil_tidx, axis=1)) / totalip) * 100, "%")

# ========================
# For time evolution 
# ========================

num_plots = 5 # number of plots minus 1 
indices = range(0, solution_lstsq_fil_tidx.shape[0], solution_lstsq_fil_tidx.shape[0] // num_plots)
n = len(list(indices))

ncols = int(np.ceil(np.sqrt(n))) #grid size is flexible
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).flatten()  # flatten in case it's 2D

for plot_idx, i in enumerate(indices):
    ax = axes[plot_idx]
    ax.plot(limiter[:, 0], limiter[:, 1], color='k')
    sc = ax.scatter(rgrid, zgrid, c=solution_lstsq_fil_tidx[i], cmap='plasma', marker='o', edgecolors='k')     
    ax.set_xlabel('r')
    ax.set_ylabel('z')
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'i = {i}')
    fig.colorbar(sc, ax=ax, label='Ip [A]')

# Hide any unused subplots
for j in range(plot_idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


## Preparing and Running JAMfit's TSVD Reconstruction Algorithm

The next solver we will use in this example notebook is the truncated singular value decomosition algorithmn also bundled in within JAMfit. 

This solver decomposes the filament and mutual inductance coupling into SVD modes, where each mode represents various jtor "current structures" that corresponds to the magnetic sensors. 

This allows us to have a high resolution/number of filaments, but then we can just solve with a smaller number of modes as long as the number of modes we pick is well below the number of sensors. 

In [ ]:
nModes = 38

To use the JAMfit tsvd algorithmn, users will first need to call the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.setup_JAMfit "prepare_tsvd_laplace()" method. The following inputs, ```sigma```, the number of shaping coils ```num_non_fil_coils```, the number of real magnetic sensors ```num_sensors```, the RZ grid in the forms ```rgrid``` and ```zgrid```, and finally the number of current modes to use for the reconstruction ```num_modes```. 

Note that having ```verbose = True``` allows for a plot of the singular values. Problems that are good for the tsvd problem typically have a sharp drop off in the singular values plot, indicating the ideal number of modes to use for this problem. However, this particular example problem does not have a sharp drop-off, indicating that tsvd still might not be ideal for this problem as we will see later. 

In [ ]:
Msc_coils_weighted, Ms_weighted, U_trun, ls_mat, ls_mat_fil, lap_proj, ip_row, N = jam_obj.prepare_tsvd_laplace(sigma, num_non_fil_coils, num_sensors, rgrid, zgrid, nModes, verbose = True)

In the following cell we run the tsvd reconstruction loop over the timesteps. 

>**Note:** that in this example, the ideal ip weighting value ```lam``` as well as the laplacian regularization factor ```lap_lam``` and the tikanov wall regularization ```reg_wall``` have already been found manually. In practical applications, sweeping through reg factors and lam weighting factors and comparing to the residual is one way for determining the ideal regularization factor. Also note that for tsvd, the number of modes ```nModes``` you pick can also heavily affect the reconstruction quality and changing the number of modes affects the ideal ```lam```, ```lap_lam```, and ```reg_wall```. 

In [ ]:
solution_tsvd_fil_tidx = [] 
solution_tsvd_wall_tidx = [] 
tsvd_Ax_tidx = [] 
tsvd_diagnostics_tidx = []

lam = 1E-3 
lap_lam = 1E-6
reg_wall = 1E-3

for t_idx in range(len(time_array_sensors)):
    Psi_at_time = magnetic_data[t_idx, :]
    ip_at_time = totalip[t_idx]
    coil_curr_at_time = coilcurrs[t_idx, :] 
    curr_expand, wall_expand, Ax, diagnostics = jam_obj.run_reconstruction_tsvd_laplace(Psi_at_time, ip_at_time, coil_curr_at_time, Msc_coils_weighted, Ms_weighted, U_trun, ls_mat, ls_mat_fil, lap_proj, ip_row, N, sigma, lam, lap_lam, reg_wall) 
    solution_tsvd_fil_tidx.append(curr_expand)
    solution_tsvd_wall_tidx.append(wall_expand)


solution_tsvd_fil_tidx = np.array(solution_tsvd_fil_tidx)
solution_tsvd_wall_tidx = np.array(solution_tsvd_wall_tidx)

## Post-Processing and Plotting Values of Interest 

Here we plot the relevant filament jtor solutions from the TSVD solver. We can see that the TSVD solver does slightly better in comparison to the LSTSQ solver in that it concentrates current closer. However, the TSVD algorithmn also tends to produce hollow body profiles as seen in the first time step. This is because the numerical algorithmn has no physical information about plasma behavior and therefore tends to favor unphysical modes that favor closer towards the outside magnetics. 

In [ ]:
# =================================
# Plotting/Processing results 
# =================================

plt.figure()
plt.plot(time_array_sensors, totalip, label='Total Ip')
plt.plot(time_array_sensors, np.sum(solution_tsvd_fil_tidx, axis=1), 'k--', label='Reconstructed Ip')
plt.xlabel('Time')
plt.ylabel('Ip [A]')
plt.ylim(0, max(np.max(totalip), np.max(np.sum(solution_tsvd_fil_tidx, axis=1))) * 1.1)
plt.legend()
plt.show()

print("Ip error percentage: ", np.mean(np.abs(totalip - np.sum(solution_tsvd_fil_tidx, axis=1)) / totalip) * 100, "%")

# ========================
# For time evolution 
# ========================

num_plots = 5 # number of plots minus 1 
indices = range(0, solution_tsvd_fil_tidx.shape[0], solution_tsvd_fil_tidx.shape[0] // num_plots)
n = len(list(indices))

ncols = int(np.ceil(np.sqrt(n))) #grid size is flexible
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axes = np.array(axes).flatten()  # flatten in case it's 2D

for plot_idx, i in enumerate(indices):
    ax = axes[plot_idx]
    ax.plot(limiter[:, 0], limiter[:, 1], color='k')
    sc = ax.scatter(rgrid, zgrid, c=solution_tsvd_fil_tidx[i], cmap='plasma', marker='o', edgecolors='k')     
    ax.set_xlabel('r')
    ax.set_ylabel('z')
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'i = {i}')
    fig.colorbar(sc, ax=ax, label='Ip [A]')

# Hide any unused subplots
for j in range(plot_idx + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()



# Post Processing 

In the following cells, we will now demonstrate how to use the \ref OpenFUSIONToolkit.ThinCurr.JAMfit.post_process_tidx "post_process_tidx()" method which utilizes tools from TokaMaker. This method obtains the last closed flux surface, the limiting points (None if they do not exist), the non-normalized psi at the last closed flux surface (LCFS), the total non-normalized psi over the entire evaluated TokaMaker grid, the safety factor, the q95 safety factor, the internal inductance, the current centroid, the geometric area centroid, and the area enclosed by the LCFS. 


There are several important inputs that are necessary to use JAMfit's postprocessing methods. Firstly "post_process_tidx()" is for a specific time step and thus requires the following inputs: 

1. The solved filaments values at the relevant timeslice from the reconstruction algorithmns ```solution_tsvd_fil_tidx```
2. The flux contribution from the wall on the full TokaMaker grid (explained further) 
3. A coil current dictionary whose keys match the TokaMaker model's coil dictionary's key **exactly** and each entry is that coil's current **at that time slice** 
4. The R grid of the filament array ```rgrid```
5. The Z grid of the filament array ```zgrid```
6. The path to the TokaMaker Model ```meshfile_tokamaker```
7. B0, the location of the magnetic center 
8. R0, the major radius
9. An OFT_env instance ```OFT_env```




In [ ]:
# this following codeblock is for the coil processing in diii-d specifically  

coil_names = ['F1A', 'F2A', 'F3A', 'F4A', 'F5A', 'F6A', 'F7A', 'F8A', 'F9A',
              'F1B', 'F2B', 'F3B', 'F4B', 'F5B', 'F6B', 'F7B', 'F8B', 'F9B',
              'ECOILA', 'ECOILB', 'E567UP', 'E567DN', 'E89DN', 'E89UP'] 


def get_diiid_dict_at_tidx(coil_dict, coilcurrs, t_idx, coil_names):
    name_to_idx = {name: i for i, name in enumerate(coil_names)}
    
    coil_curr_dict = OrderedDict()
    for coil in coil_dict.keys():
        base_name = re.sub(r'\d+$', '', coil)   # strip trailing digits
        idx = name_to_idx[base_name]
        coil_curr_dict[base_name] = coilcurrs[t_idx, idx]
    return coil_curr_dict


JAMfit's reconstruction process does not solve for wall currents directly, but rather in the form of **current potentials** i.e the set of weights for each mode used to represent the wall current. In our case, the wall is represented with 4 modes that we selected from the vde_diiid_create_ex.ipynb notebook. To turn these current potential weights into usable psi (flux) values for the wall's contribution to the plasma, we convert the potential solution into flux. We do this through placing a flux loop (these arent physically real sensors but for helping to map the potential to flux to that point in space) at every point inside the limiter, then compute the flux each one sees from the wall by multiply the ```Ms``` mutual inductance matrix that couples sensors to the wall modes with the wall mode potential weights that we solved for. Remember that the ```Ms``` mutual inductance matrix couples ***every defined sensor*** to the wall modes. 
```
wall_psi = solution_wall_at_time @ Ms / (2*pi)
```
This is done through \ref OpenFUSIONToolkit.ThinCurr.JAMfit.get_wall_psi "get_wall_psi()" method.


Placing a flux loop at every TokaMaker grid point (20k+) would be too computationally expensive and unnecessary as we only care about the wall's contriution to where the plasma is located, which would be inside the limiter. We can find every TokaMaker point inside the limiter using \ref OpenFUSIONToolkit.ThinCurr.JAMfit.get_inside_limiter_pts "get_inside_limiter_pts()" methodand adjust the sparsity of how many loops we place by adjusting the "stride" parameter, in our case ```stride = 2```. 

>**Note:** Note: The flux loops need to already be initialized as part of your magnetic sensor array. In our case, this was done for you when floops.loc was loaded in the previous notebook. That's why we track ```num_sensors```, it lets us separate these "virtual" flux loops from the real sensors that are actually measuring physical data in the reconstruction.

In [ ]:
inside_lim_pts, sparse_mask, inside_mask, fullgrid = get_inside_limiter_pts(meshfile_tokamaker, myOFT, stride=2, verbose=True)

Note that this loop can take some time, upwards of up to 6 minutes. 

In [ ]:
lcfs_pts_tidx = [] 
limiting_pts_tidx = []
q95_tidx = [] 
internal_inductance_tidx = []
current_centroid_tidx = []
area_tidx = []
total_psi_tidx = [] 

for t_idx in range(len(time_array_sensors)):
    filaments_at_time = solution_tsvd_fil_tidx[t_idx]

    wall_psi_sparse = jam_obj.get_wall_psi_tidx(num_sensors, solution_tsvd_wall_tidx[t_idx])
    wall_psi_full_grid_at_time = np.zeros(fullgrid.shape[0])
    wall_psi_full_grid_at_time[sparse_mask] = wall_psi_sparse  # known values
    fill_mask = inside_mask & ~sparse_mask     # fill the skipped-over inside points by interpolating from the sparse ones
    wall_psi_full_grid_at_time[fill_mask] = griddata(
        fullgrid[sparse_mask][:, :2], wall_psi_sparse, fullgrid[fill_mask][:, :2], method='linear'
    )
    nan_mask = fill_mask & np.isnan(wall_psi_full_grid_at_time) # catch any NaNs (points outside convex hull of sparse set) with nearest-neighbor
    if np.any(nan_mask):
        wall_psi_full_grid_at_time[nan_mask] = griddata(
            fullgrid[sparse_mask][:, :2], wall_psi_sparse, fullgrid[nan_mask][:, :2], method='nearest'
        )

    coil_curr_dict = get_diiid_dict_at_tidx(coil_dict, coilcurrs, t_idx, coil_names) 
    lcfs_pts, lim_pts, _, total_psi, _, q95, li, curr_cent, _, area = jam_obj.post_process_tidx(filaments_at_time, wall_psi_full_grid_at_time, coil_curr_dict, rgrid, zgrid, verbose=False)
    lcfs_pts_tidx.append(lcfs_pts)
    limiting_pts_tidx.append(lim_pts)
    total_psi_tidx.append(total_psi)
    q95_tidx.append(q95)
    internal_inductance_tidx.append(li)
    current_centroid_tidx.append(curr_cent)
    area_tidx.append(area)
clear_output()

Here we plot our results of the total psi for specified time steps. 

In [ ]:
import matplotlib.tri as mtri
selected_idx = [0, 8, 40, 60, 80, 100]
selected = [(j, total_psi_tidx[j], limiting_pts_tidx[j], current_centroid_tidx[j]) for j in selected_idx if limiting_pts_tidx[j] is not None]
ncols = 5
nrows = 1
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3, nrows*4))
axes = axes.flatten()
for ax, (j, total_psi, limiting_pts, cent_signed) in zip(axes, selected):
    ax.scatter(rgrid, zgrid, color='#ADD8E6', s=10, zorder=0, alpha=0.6)

    # Build triangulationover the 2592 nearest-neighbor grid points
    triang_orig = mtri.Triangulation(mygs.r[:, 0], mygs.r[:,1], mygs.lc[:, :3])
    cf0 = ax.tricontourf(triang_orig, total_psi, levels=100, cmap='RdBu_r')
    ax.tricontour(triang_orig, total_psi, levels=20, colors='k', linewidths=0.5, alpha=0.4) 

    if limiting_pts[0] is not None and limiting_pts[0][0] is not None:
        ax.scatter(limiting_pts[0], limiting_pts[1], label='Limiting', marker='x', color='r', s=60, zorder=3)
    
    ax.scatter(cent_signed[0], cent_signed[1], label='Current Centroid', marker='s', color='g', s=60, zorder=4)
    ax.set_aspect('equal')
    ax.set_title(f'idx = {j}, t={time_array_sensors[j]:.4f} s', fontsize=8)  
    ax.legend(loc = 'lower center', fontsize=6, framealpha= 1)
    ax.plot(limiter[:,0], limiter[:,1], 'k-', zorder =1)


plt.tight_layout()
plt.show()